In [1]:
import os

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import scanpy as sc
import torch

import T_perturb
from T_perturb.Dataloaders.datamodule import CellGenDataModule
from T_perturb.Perturb.trainer import PerturberTrainer

from T_perturb.src.utils import label_encoder
from T_perturb.tests.test_cellgen_training import dummy_dataset
from T_perturb.tests.test_countdecoder_training import dummy_cell_gene_matrix

/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed_no = 42
pl.seed_everything(seed_no)
torch.manual_seed(seed_no)

Seed set to 42


In [3]:
if os.getcwd().split('/')[-1] != 'healthy_imm_expr':
    # set working directory to root of repository
    os.chdir('/lustre/scratch126/cellgen/team361/kl11/t_generative/')

In [4]:
tgt_vocab_size = 100  # +1 for padding token
num_samples = 100
num_genes = 100
max_seq_length = 50
n_total_tps = 2
num_samples = 100
batch_size = 4
pred_tps = [1, 2]
context_tps = [1, 2]
d_model = 12

genes_to_perturb = ['ISG15']
perturbation_token = 1


In [9]:
src_counts = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
)
src_dataset = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
)
tgt_counts_dict = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
    total_time_steps=n_total_tps,
)
tgt_datasets = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
    total_time_steps=n_total_tps,
)

In [10]:
# check if cuda is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [59]:
import importlib
import T_perturb.src.utils
importlib.reload(T_perturb.Perturb.trainer)
importlib.reload(T_perturb.src.utils)
from T_perturb.Perturb.trainer import PerturberTrainer

trainer_params = {
    'tgt_vocab_size': tgt_vocab_size,
    'd_model': d_model,
    'num_heads': 4,
    'num_layers': 1,
    'd_ff': 8,
    'max_seq_length': max_seq_length + 10,
    'dropout': 0.0,
    'pred_tps': pred_tps,
    'context_tps': context_tps,
    'n_total_tps': n_total_tps,
    'precision': 'high',
    'mask_scheduler': 'pow',
    'mapping_dict_path': 'T_perturb/T_perturb/pp/res/hspc/token_id_to_genename_10000_hvg.pkl',
    'output_dir': './T_perturb/T_perturb/tests/res',
    'encoder': 'Transformer_encoder',
    'var_list': None,
    'genes_to_perturb': genes_to_perturb,
    # 'gene_module_list': [
    #     'TNFRSF18', 'TNFRSF4','C1QTNF12', 
    #     'TAS1R3', 'ANKRD65', 'VWA1', 
    #     'ATAD3C', 'TMEM240', 'MIB2'
    #     ],
    'num_of_background_genes': 10,
    'perturbation_mode': 'delete',
    'perturbation_sequence': 'tgt',
    'validation_mode': 'inference',
    'context_mode': True,
    'temperature':1.5,
    'iterations': 19,
    'sequence_length': max_seq_length - 10,
    'pos_encoding_mode': 'time_pos_sin',
    'return_attn': False,
    'mask_scheduler': 'cosine',
    'var_list': ['cell_type']
}
decoder_module = PerturberTrainer(
    **trainer_params
)
data_module = CellGenDataModule(
    batch_size=batch_size,
    src_counts=src_counts,
    tgt_counts_dict=tgt_counts_dict,
    src_dataset=src_dataset,
    tgt_datasets=tgt_datasets,
    num_workers=1,
    pred_tps=pred_tps,
    context_tps=context_tps,
    n_total_tps=n_total_tps,
    split=False,
    max_len=max_seq_length,
    var_list=['cell_type']
)


Using CPU for training
cls_token_1 tensor([100])
cls_token_2 tensor([101])
{4: 'PERM1', 5: 'HES4', 6: 'ISG15', 7: 'RNF223', 8: 'TNFRSF18', 9: 'TNFRSF4', 10: 'C1QTNF12', 11: 'TAS1R3', 12: 'ANKRD65', 13: 'VWA1', 14: 'ATAD3C', 15: 'TMEM240', 16: 'MIB2', 17: 'SLC35E2B', 18: 'PRKCZ', 19: 'SKI', 20: 'PLCH2', 21: 'HES5', 22: 'TNFRSF14', 23: 'TTC34', 24: 'PRDM16', 25: 'SMIM1', 26: 'CEP104', 27: 'AJAP1', 28: 'PLEKHG5', 29: 'UTS2', 30: 'ERRFI1', 31: 'SLC45A1', 32: 'RERE', 33: 'SLC2A5', 34: 'GPR157', 35: 'H6PD', 36: 'SPSB1', 37: 'PIK3CD', 38: 'NMNAT1', 39: 'RBP7', 40: 'KIF1B', 41: 'CASZ1', 42: 'FBXO2', 43: 'FBXO44', 44: 'MTHFR', 45: 'TNFRSF8', 46: 'TNFRSF1B', 47: 'VPS13D', 48: 'C1orf158', 49: 'LRRC38', 50: 'PDPN', 51: 'PRDM2', 52: 'FHAD1', 53: 'CELA2B', 54: 'DNAJC16', 55: 'SLC25A34', 56: 'FBLIM1', 57: 'FAM131C', 58: 'EPHA2', 59: 'SPATA21', 60: 'NBPF1', 61: 'CROCC', 62: 'MFAP2', 63: 'PADI4', 64: 'NBL1', 65: 'HTR6', 66: 'TMCO4', 67: 'RNF186', 68: 'PLA2G2A', 69: 'PLA2G2D', 70: 'CAMK2N1', 71: 'CDA', 

In [60]:
accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
trainer = pl.Trainer(
    limit_test_batches=5,  # Limit to a single batch for quick testing
    logger=False,
    accelerator=accelerator,
    devices=1,  # inference only on one gpu
    precision=32,
)
trainer.test(
    decoder_module, 
    data_module,
    )


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Testing DataLoader 0:   0%|          | 0/5 [00:00<?, ?it/s]perturb tensor([6])
tensor([[100,   4,  39,  58,  52,  32,   6,   5,  24,  20,  91,  30,  98,  60,
          43,  10,  13,  12,  54,  67,   8,  65,  61,  78,  31,  16,  74,  28,
          97,  26,   9,  70,  50,  34,  71,  14,  59,  99,  33,  49,  23,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0],
        [100,   9,  90,  41,  18,  67,  56,  51,  61,  99,  69,  70,  11,  20,
          91,  43,  17,  63,  34,  27,  54,  52,  23,  40,  87,  58,  72,  49,
          16,  24,  38,  84,  86,  31,  92,  15,  47,  66,  25,  55,  22,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0],
        [100,  76,  99,  53,  80,  90,  93,  64,  59,  49,  83,  70,  21,  38,
          58,  44,  12,  45,  82,  62,  97,  42,  78,  71,  56,  95,  88,  50,
          84,  81,  92,   8,   3,  68,  89,   5,  16,  14,  94,   6,  60,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0],
        [100,  84,  80,  24,  27,  31,   2,  

/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


RuntimeError: No active exception to reraise